In [1]:
import hashlib
import os
import secrets


class CHAPServer:

    def __init__(self, client_username, shared_secret):
        self.username = client_username
        self._secret = shared_secret  # Secret stored securely on the server
        self.current_id = None
        self.current_challenge = None

    def generate_challenge(self):
        """Step 1: Server generates a unique ID and a random challenge."""
        # A 1-byte identifier (0-255)
        self.current_id = secrets.randbelow(256).to_bytes(1, byteorder="big")
        # A random 16-byte cryptographic challenge string
        self.current_challenge = secrets.token_bytes(16)

        print(f"[SERVER] Generated ID: {self.current_id.hex()}")
        print(f"[SERVER] Generated Challenge: {self.current_challenge.hex()}")

        return self.current_id, self.current_challenge

    def verify_response(self, client_response):
        """Step 3: Server calculates its own expected hash and verifies the client."""
        # Create the exact same byte block: ID + Secret + Challenge
        hasher = hashlib.md5()
        hasher.update(self.current_id)
        hasher.update(self._secret.encode("utf-8"))
        hasher.update(self.current_challenge)
        expected_response = hasher.digest()

        print(f"[SERVER] Expected Response Hash: {expected_response.hex()}")
        print(f"[SERVER] Received Client Hash:   {client_response.hex()}")

        # Clean up challenge data immediately to prevent replay attempts
        self.current_id = None
        self.current_challenge = None

        if secrets.compare_digest(client_response, expected_response):
            print("[SERVER] Match! AUTHENTICATION SUCCESSFUL.\n")
            return True
        else:
            print("[SERVER] Mismatch! AUTHENTICATION FAILED.\n")
            return False


class CHAPClient:

    def __init__(self, username, shared_secret):
        self.username = username
        self._secret = shared_secret  # Secret known to the client

    def calculate_response(self, chap_id, challenge):
        """Step 2: Client mixes ID, Secret, and Challenge to generate the MD5 hash."""
        hasher = hashlib.md5()

        # Input data order matters strictly: ID + Secret + Challenge
        hasher.update(chap_id)
        hasher.update(self._secret.encode("utf-8"))
        hasher.update(challenge)

        client_hash = hasher.digest()
        print(f"[CLIENT] Calculated Response Hash: {client_hash.hex()}")
        return client_hash


# ==========================================
# SIMULATION RUNS
# ==========================================
if __name__ == "__main__":
    # The true credential shared between both parties out-of-band
    TRUE_PASSWORD = "EveryGoodAndPerfectGiftIsFromAbove"

    print("--- SCENARIO 1: Valid Login Attempt ---")
    server = CHAPServer(client_username="initiator_01", shared_secret=TRUE_PASSWORD)
    client_legit = CHAPClient(
        username="initiator_01", shared_secret=TRUE_PASSWORD
    )

    # 1. Server sends challenge out
    chap_id, challenge = server.generate_challenge()

    # 2. Client computes response based on the challenge data
    response_hash = client_legit.calculate_response(chap_id, challenge)

    # 3. Server validates it
    server.verify_response(response_hash)

    print("--- SCENARIO 2: Malicious/Wrong Password Attempt ---")
    # Attacker tries to log in using an incorrect password guessing attempt
    client_attacker = CHAPClient(
        username="initiator_01", shared_secret="WrongPasswordGuess"
    )

    # 1. Server sends a fresh new challenge
    chap_id, challenge = server.generate_challenge()

    # 2. Attacker computes a response using their wrong secret
    attacker_hash = client_attacker.calculate_response(chap_id, challenge)

    # 3. Server catches the discrepancy
    server.verify_response(attacker_hash)

--- SCENARIO 1: Valid Login Attempt ---
[SERVER] Generated ID: 93
[SERVER] Generated Challenge: f5cb84067ff95b4650f2c7686b142263
[CLIENT] Calculated Response Hash: b25bc353c5fd0af64681d3b9ac056c6c
[SERVER] Expected Response Hash: b25bc353c5fd0af64681d3b9ac056c6c
[SERVER] Received Client Hash:   b25bc353c5fd0af64681d3b9ac056c6c
[SERVER] Match! AUTHENTICATION SUCCESSFUL.

--- SCENARIO 2: Malicious/Wrong Password Attempt ---
[SERVER] Generated ID: e2
[SERVER] Generated Challenge: 981bbf34094b77b2a34adca1a399eed2
[CLIENT] Calculated Response Hash: 496674c2faf3f8f6bd65b281df773c39
[SERVER] Expected Response Hash: db8561f280a0d78e87fe2eacd57272ce
[SERVER] Received Client Hash:   496674c2faf3f8f6bd65b281df773c39
[SERVER] Mismatch! AUTHENTICATION FAILED.

